# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/',
 'https://edwarddonner.com/2025/09/1

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/
https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/

In [7]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [8]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'linkedin page', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'twitter profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'facebook page',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [9]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [10]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'LinkedIn profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook profile',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [11]:
select_relevant_links("https://huggingface.co")

{'links': [{'type': 'homepage', 'url': 'https://huggingface.co/'},
  {'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'blog page', 'url': 'https://huggingface.co/blog'},
  {'type': 'GitHub page', 'url': 'https://github.com/huggingface'},
  {'type': 'LinkedIn page',
   'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'Twitter page', 'url': 'https://twitter.com/huggingface'},
  {'type': 'Discourse community', 'url': 'https://discuss.huggingface.co'},
  {'type': 'Discord invite', 'url': 'https://huggingface.co/join/discord'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [12]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [13]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Community
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
moonshotai/Kimi-K2.5
Updated
1 day ago
•
53.5k
•
1.36k
Tongyi-MAI/Z-Image
Updated
4 days ago
•
4.76k
•
761
tencent/HunyuanImage-3.0-Instruct
Updated
4 days ago
•
116
•
748
nvidia/personaplex-7b-v1
Updated
4 days ago
•
83.8k
•
1.57k
deepseek-ai/DeepSeek-OCR-2
Updated
3 days ago
•
103k
•
604
Browse 2M+ models
Spaces
Running
on
Zero
Featured
1.1k
Qwen3-TTS Demo
🎙
1.1k
Generate realistic speech from text with custom voices or voice cloning
Running
on
Zero
MCP
1.93k
Z Image Turbo
🖼
1.93k
Generate stunning AI images from text descriptions in seconds
Running
on
Zero
Featured
1.29k
Qwen Image Multiple Angles 3D Camera
🎥
1.29k
Adjust came

In [14]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [15]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [16]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nCommunity\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nmoonshotai/Kimi-K2.5\nUpdated\n1 day ago\n•\n53.5k\n•\n1.36k\nTongyi-MAI/Z-Image\nUpdated\n4 days ago\n•\n4.76k\n•\n761\ntencent/HunyuanImage-3.0-Instruct\nUpdated\n4 days ago\n•\n116\n•\n748\nnvidia/personaplex-7b-v1\nUpdated\n4 days ago\n•\n83.8k\n•\n1.57k\ndeepseek-ai/DeepSeek-OCR-2\nUpdated\n3 days ago\n•\n103k\n•\n604\nBrowse 2M+ models\nSpaces\nRunning\non\nZero\nFeatured\n1.1k\nQwen3

In [17]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [18]:
create_brochure("HuggingFace", "https://huggingface.co")

# Hugging Face Brochure

---

## Welcome to Hugging Face
**The AI community building the future.**

Hugging Face is the premier collaboration platform for the global machine learning (ML) community. It is the central hub where ML engineers, data scientists, and developers come together to share, explore, and innovate with open-source models, datasets, and applications across all AI modalities – including text, image, video, audio, and 3D.

---

## What We Offer

### 1. **Massive Repository of Models, Datasets & Apps**
- Browse **2 million+ models** shared by the community.
- Access **500k+ datasets** vital for training and evaluation.
- Explore over **1 million AI applications** built and shared openly.

We enable the AI ecosystem to move faster by providing:
- Unlimited hosting and collaboration on public ML models.
- Tools to discover and build unique AI-powered applications.
  
### 2. **Spaces – AI Apps Made Easy**
A simple platform to create and showcase AI demos and apps such as:
- Realistic Text-to-Speech Generators.
- AI image creation and editing from text prompts.
- Video generation from images using AI.
Spaces allow effortless sharing and deployment of AI applications with a supportive community.

### 3. **Accelerate Your Machine Learning**
- Access to **Hugging Face's open-source stack** — empowering faster development and experimentation.
- Paid compute resources to scale your machine learning projects.

---

## Our Community & Culture
Hugging Face is **more than a tech platform — it is a thriving open and ethical AI community** where members:
- Collaborate on open-source machine learning projects.
- Build professional ML portfolios to showcase their technical contributions.
- Learn and exchange ideas that drive the future of AI.

We believe in transparency, openness, and advancing AI together in a responsible, ethical way.

---

## Who Uses Hugging Face?
- **Machine Learning Engineers** and **Data Scientists** discovering models for research.
- **AI Researchers** contributing new architectures and datasets.
- **Developers** building and deploying impactful AI applications.
- **Enterprises** seeking to accelerate AI innovation with accessible ML tools.
- Open-source enthusiasts and AI practitioners worldwide.

---

## Careers at Hugging Face
Join a dynamic, mission-driven team dedicated to democratizing AI. As part of Hugging Face, you will:
- Work alongside ML experts and innovators.
- Influence the future of collaborative AI development.
- Enjoy a culture of learning, openness, and impact.

Visit our careers page to explore current job openings and become part of the AI revolution.

---

## Connect with Hugging Face
- **Website:** https://huggingface.co
- **Community Hub:** Collaborate, discuss, and grow your machine learning career.
- **Sign Up:** Create your profile, share your projects, and accelerate your AI journey.

---

## Brand Highlights 
- Vibrant and recognizable brand colors: Yellow #FFD21E and Orange #FF9D00.
- Trusted by a fast-growing, global AI community.
- Positioned at the forefront of open-source machine learning advancement.

---

## Join Us in Building the Future of AI at Hugging Face!

**Collaborate. Innovate. Share.**

---

*Hugging Face – The Home of Machine Learning*

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [21]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [22]:
stream_brochure("HuggingFace", "https://huggingface.co")

# Hugging Face Brochure

---

## About Hugging Face

**Hugging Face** is the AI community building the future of machine learning. It is a unique platform where the global machine learning community collaborates on models, datasets, and AI applications to accelerate innovation and development across all modalities including text, image, video, audio, and 3D.

The Hugging Face Hub serves as a central place for sharing, discovering, and experimenting with open-source ML resources. It empowers machine learning engineers, scientists, and end users to learn, collaborate, and contribute to building an open, responsible, and ethical AI future together.

---

## What Hugging Face Offers

- **Models:** Browse over 2 million machine learning models ranging from state-of-the-art language models to specialized AI vision and speech models.
- **Datasets:** Access a vast collection of more than 500,000 datasets used to train and evaluate machine learning algorithms.
- **Spaces:** Deploy and explore over 1 million AI applications hosted on the platform, showcasing innovative projects in AI.
- **Community:** Join a fast-growing global community contributing to open-source ML, sharing knowledge, feedback, and expertise.
- **Enterprise Solutions:** Tailored AI solutions for businesses to integrate and accelerate machine learning workflows at scale.
- **Open Source Stack:** Leverage a comprehensive open-source toolkit designed for faster, flexible, and ethical AI development.

---

## Why Choose Hugging Face?

- **Collaboration at Scale:** Host unlimited public models, datasets, and applications and collaborate with ML practitioners worldwide.
- **Multi-modal Exploration:** Supports AI development across diverse data types and applications including text, images, videos, audio, and 3D.
- **Community-driven Innovation:** The platform fuels innovation by providing tools and infrastructure for ML researchers, developers, and companies to build together.
- **Portfolio Building:** Share your work publicly and build your professional profile in the machine learning community.
- **Ethical AI Commitment:** Hugging Face promotes the responsible development and deployment of AI technologies through openness and transparency.

---

## Customers and Use Cases

Hugging Face is widely used by:

- AI researchers and data scientists experimenting with state-of-the-art ML models.
- Developers building AI applications with easy deployment on the platform’s Spaces.
- Businesses seeking enterprise-grade AI solutions and API integrations.
- Educational institutions fostering AI learning and research.
- Open-source contributors advancing AI technology and ethics.

Some of the prominent projects and models trending on Hugging Face include advanced text-to-speech demos, image generation tools, video creation AI, and multilingual translation models.

---

## Company Culture and Community

Hugging Face fosters a culture of:

- **Open Collaboration:** Encouraging transparency and sharing in AI development.
- **Inclusivity:** Creating an accessible and diverse environment for all contributors.
- **Innovation:** Driving forward cutting-edge AI research and applications.
- **Ethics and Responsibility:** Committing to building AI that aligns with human values and social good.

Community members are encouraged to contribute code, datasets, and AI applications while receiving support and recognition through the platform.

---

## Careers at Hugging Face

Hugging Face offers exciting career opportunities for:

- Machine Learning Engineers and Researchers
- Software Developers with passion for AI
- Data Scientists focused on open source innovation
- Community Managers and Advocates
- Product and Design professionals in AI tools

Join Hugging Face to be at the forefront of machine learning, work in a collaborative and inclusive environment, and contribute to building the future of AI.

---

## Get Involved

- **Explore AI apps, models, and datasets** at [huggingface.co](https://huggingface.co)
- **Sign up** to share your work and build your ML portfolio
- **Join the community** and contribute to an open and ethical AI future

---

**Hugging Face**  
*The AI community building the future.*

---

### Brand Colors
- Yellow: #FFD21E  
- Orange: #FF9D00  
- Gray: #6B7280  

---

Embrace the future of AI with Hugging Face — where collaboration meets innovation.

In [23]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

# Hugging Face: The AI Community Building the Future

---

## About Hugging Face

Hugging Face is a collaborative platform dedicated to advancing the future of artificial intelligence. It serves as the home for the machine learning (ML) community to create, discover, and innovate together on models, datasets, and applications. The platform hosts over 2 million models, 500,000 datasets, and 1 million+ AI-powered applications, making it one of the largest hubs for open-source ML resources.

---

## Our Mission

Hugging Face empowers ML engineers, researchers, scientists, and developers worldwide to collaborate and share their work openly, fostering an inclusive and ethical AI ecosystem. By offering a centralized place to host and explore cutting-edge projects, we accelerate progress in all AI modalities including text, image, video, audio, and 3D.

---

## Platform Highlights

- **Models:** Access and contribute to an extensive library of 2M+ publicly available machine learning models, such as text generators, image classifiers, and speech synthesis models.
- **Datasets:** Discover and share high-quality datasets to train and improve ML solutions—browse from over 500k datasets covering diverse domains.
- **Spaces:** Deploy and interact with AI applications live on the platform, including demos for speech generation, image editing, and 3D video creation.
- **Community:** Collaborate with a fast-growing community of ML experts and enthusiasts, sharing insights and solutions to push the state of AI technology.
- **Open-source Stack:** Utilize Hugging Face’s open-source libraries to speed up your ML workflows and build your personal or enterprise projects faster.
- **Enterprise Solutions:** Customized AI offerings geared towards corporate needs for scalable and ethical AI integration.

---

## Company Culture

Hugging Face fosters an open, inclusive, and ethical AI community culture. It is a space where innovation is fueled by collaboration and transparency, encouraging users and contributors to learn, co-create, and share their AI developments freely. The company values openness, diversity, and community-driven progress, aiming to build trustworthy AI that benefits everyone.

---

## Who Uses Hugging Face?

Our platform supports a broad spectrum of users including:

- Machine learning engineers and data scientists developing state-of-the-art models.
- AI researchers innovating in natural language processing (NLP), computer vision, speech, and more.
- Developers seeking to integrate AI capabilities into their applications.
- Enterprises aiming to deploy ethical and cutting-edge AI solutions.
- Educators and students leveraging open resources for teaching and learning ML.

---

## Careers and Opportunities

Hugging Face is continuously seeking passionate individuals to join and contribute to the AI revolution. Whether you're an AI researcher, software engineer, community manager, or product designer, you can find exciting roles where your expertise will drive real impact on a global scale.

---

## Get Involved

- **Sign Up:** Create your profile, share your projects, and build your ML portfolio on Hugging Face.
- **Explore:** Dive into millions of open-source ML models, datasets, and applications.
- **Collaborate:** Join a thriving community to discuss, innovate, and create the future of AI.
- **Enterprise:** Contact us for solutions customized to accelerate your AI journey.

---

## Contact & Links

- Website: [huggingface.co](https://huggingface.co)
- Explore Models & Apps
- Join the Community
- Career Opportunities

---

## Brand Colors & Logos

- **Primary Colors:** Yellow (#FFD21E), Orange (#FF9D00), and Grey (#6B7280)  
- Logos available in SVG, PNG, and AI formats for branding and partnership purposes.

---

*Hugging Face is proud to be at the forefront of building an open, collaborative, and ethical AI future. Join us and be part of the movement.*

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>